In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np
import re
import cmath
import math
import scipy.io
import matplotlib.pyplot as plt 
from scipy import signal
import matplotlib.mlab as mlab
import matplotlib.gridspec as gridspec


infile_path = 'Doherty_scaled.csv'
df = pd.read_csv(infile_path)

# Extract the 'xI_real' and 'xI_imag' columns

xin_real = df['xI_real']
xin_imag = df['xI_imag']
yout_real = df['yout_real']
yout_imag = df['yout_imag']


In [2]:
t_ns = np.arange(1,120001)

# read real, imaginary parts and calculate amplitude
xin = xin_real + 1j*xin_imag
xin_amp = abs(xin)
#xin_phase = phase(xin)
xinn = xin/max(xin_amp)
xinn_amp = abs(xinn)

yout = yout_real + 1j*yout_imag
yout_amp = abs(yout)
# normalize output to peak value
youtn = yout/max(yout_amp)
youtn_amp = abs(youtn)
#yout_phase = cmath.phase(y_out)

# compute rms value for each signal
xin_rms = np.sqrt(np.mean(np.abs(xin)**2))
yout_rms = np.sqrt(np.mean(np.abs(yout)**2))
gainrms_dB = 20*math.log10(yout_rms/xin_rms)
print(gainrms_dB)

y_arr = np.column_stack((xin_real, xin_imag))
youtn = np.array(youtn)
youtn_real = youtn.real
youtn_imag = youtn.imag
X_arr = np.column_stack((youtn_real, youtn_imag))

X = pd.DataFrame(X_arr, columns=['yout_real', 'yout_imag'])
y = pd.DataFrame(y_arr, columns=['xin_real', 'xin_imag'])

11.096999415866108


In [3]:
X_new = X.iloc[90000:120001]
print("Shape of X_new:", X_new.shape)
# Keep the first 90,000 rows in both dataframes:
X = X.iloc[:90000]
y = y.iloc[:90000]

print("Shape of y:", y.shape)
print("Shape of X:", X.shape)


Shape of X_new: (30001, 2)
Shape of y: (90000, 2)
Shape of X: (90000, 2)


In [4]:
#NN model
model = Sequential([
    Dense(32, activation='relu', input_shape=(2,)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(2)  # Output layers
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

# Train the model
history = model.fit(X, y, epochs=12, validation_split=0.2, verbose=1)

print("Training complete")


# Function to parse complex numbers, including those with scientific notation from excel csv file
def parse_complex_number(s):
    s = s.replace("i", "j")  
    # Updated regex to handle scientific notation
    match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
    if match:
        real_part = float(match.group(1))
        imag_part = float(match.group(2))
        return real_part, imag_part
    else:
        raise ValueError(f"Malformed complex number: {s}")


# Predict using the model
y_new_pred = model.predict(X_new)  # Ensure this returns a 2D array

# Create the DataFrame with predictions
df_predict = pd.DataFrame({
    'xI_predict_real': y_new_pred[:, 0],
    'xI_predict_imag': y_new_pred[:, 1]
})

# Create complex numbers
df_predict['xI_predict'] = (df_predict['xI_predict_real'] + 1j * df_predict['xI_predict_imag']).astype(np.complex128)

# Prepare data for MATLAB file generation
data_to_save = {'xI_predict': df_predict['xI_predict'].values}


#Change this to your working directory
output_mat_file_path = 'C:\\Users\\Luke McCubbin\\Box\\NN PA DPD\\Doherty_Sravya\\Post_distortion_doherty_results.mat'
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f'Predictions saved to {output_mat_file_path}')

Epoch 1/12


D:\Apps\Anaconda\Anaconda\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2250/2250 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9521 - loss: 0.1843 - val_accuracy: 0.9815 - val_loss: 0.0113
Epoch 2/12
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9791 - loss: 0.0118 - val_accuracy: 0.9807 - val_loss: 0.0109
Epoch 3/12
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9790 - loss: 0.0116 - val_accuracy: 0.9817 - val_loss: 0.0109
Epoch 4/12
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9788 - loss: 0.0116 - val_accuracy: 0.9798 - val_loss: 0.0127
Epoch 5/12
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9793 - loss: 0.0114 - val_accuracy: 0.9814 - val_loss: 0.0106
Epoch 6/12
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9806 - loss: 0.0114 - val_accuracy: 0.9807 - val_loss: 0.0122
Epoch 7/12
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9786 - loss: 0.0115 - val_accuracy: 0.9823 - val_loss: 0.0104
Epoch 8/12
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9810 - loss: 0.0111 - val_accurac